### **Detectando Idioma**

Este notebook tem a finalidade de detectar o idioma da concatenação do título + conteúdo de cada post. Além de agregalo ao dataset

---

In [2]:
import os
import json
import pandas as pd
from langdetect import detect, DetectorFactory
import numpy as np
from tqdm import tqdm
import pandas as pd
from multiprocessing import Pool, cpu_count

In [3]:
rows = []
erros = []

# Itera por cada profundidade
for depth in range(3 + 1):
    dir_ = f"../data/raw/expansao/{depth}/subreddits"
    if not os.path.exists(dir_):
        continue
    for filename in os.listdir(dir_):
        if filename.endswith(".json"):
            path = os.path.join(dir_, filename)
            try:
                with open(path, "r", encoding="utf-8") as f:
                    posts = json.load(f)
                # Adiciona a profundidade em cada post para rastreabilidade
                for post in posts:
                    post["depth"] = depth
                rows.extend(posts)
            except json.JSONDecodeError:
                erros.append(path)


df = pd.DataFrame(rows)

print(f"Arquivos com erro: {len(erros)}")
print(f"Quantidade de subreddits únicos: {df['subreddit'].nunique()}")
print(f"Quantidade de autores únicos: {df['author'].nunique()}")
print(f"Quantidade de posts carregados: {len(df)}")

Arquivos com erro: 0
Quantidade de subreddits únicos: 7372
Quantidade de autores únicos: 1202164
Quantidade de posts carregados: 2227113


In [4]:
print(f"Número de nulos no título: {df['title'].isnull().sum()}")
print(f"Número de nulos no conteúdo: {df['selftext'].isnull().sum()}")

Número de nulos no título: 0
Número de nulos no conteúdo: 1827562


In [5]:
df[df['title'] == '[removed]']

,id,title,selftext,author,subreddit,score,timestamp,num_comments,url,is_self,over_18,removed_by_category,depth
176891,v3zhbg,[removed],NaN,Idonthavefriendss,shitposting,60496,2022-06-03T13:31:36+00:00,591,https://reddit.com/r/shitposting/comments/v3zh...,False,False,NaN,0
362815,ax4use,[removed],Should’ve read the description better,okaywhat22,antijokes,3203,2019-03-04T08:30:51+00:00,54,https://reddit.com/r/AntiJokes/comments/ax4use...,True,False,NaN,1
596874,tc9pu5,[removed],NaN,chilre_illa,sakkath,55,2022-03-12T05:52:12+00:00,15,https://reddit.com/r/sakkath/comments/tc9pu5/r...,False,False,NaN,1
1547483,fvnkkm,[removed],NaN,RetroAmericano,memes_of_the_dank,9280,2020-04-05T23:15:10+00:00,91,https://reddit.com/r/Memes_Of_The_Dank/comment...,False,False,NaN,1
2123973,hqvpan,[removed],NaN,Nearby-Airport,redditmoment,12443,2020-07-14T05:31:15+00:00,181,https://reddit.com/r/redditmoment/comments/hqv...,False,False,NaN,1


In [6]:
df[df['selftext'] == '[removed]']

,id,title,selftext,author,subreddit,score,timestamp,num_comments,url,is_self,over_18,removed_by_category,depth


In [5]:
df['selftext'] = df['selftext'].fillna('')
df['title'] = df['title'].fillna('')

df['text'] = (df["title"].fillna("") + " " + df["selftext"].fillna("")).str.strip()

In [7]:
tqdm.pandas()
DetectorFactory.seed = 0

def detect_language(text):
    if not isinstance(text, str) or text.strip() == "":
        return "unknown"
    try:
        return detect(text)
    except:
        return "unknown"
    
def parallel_detect(texts, num_workers=None):
    if num_workers is None:
        num_workers = cpu_count() 
    with Pool(num_workers) as pool:
        results = pool.map(detect_language, texts)
    return results

df["lang"] = parallel_detect(df["text"].tolist())

In [8]:
print("Contagem por idioma:")
df['lang'].value_counts().head(15)

Contagem por idioma:


lang
en         1832330
de           35566
pt           25392
af           24031
tl           23641
da           23156
es           21940
nl           19322
fr           19222
it           18365
no           15968
unknown      15791
id           15369
cy           14872
so           14204
Name: count, dtype: int64

In [9]:
print("\nPorcentagem por idioma:")
df['lang'].value_counts(normalize=True).head(15) * 100


Porcentagem por idioma:


lang
en         82.273778
de          1.596955
pt          1.140131
af          1.079020
tl          1.061509
da          1.039732
es          0.985132
nl          0.867581
fr          0.863090
it          0.824610
no          0.716982
unknown     0.709035
id          0.690086
cy          0.667770
so          0.637776
Name: proportion, dtype: float64

### Tabela de Distribuição de Idiomas Detectados

| Código | Idioma | Proporção (%) | Status / Observação | Ação Recomendada |
| :--- | :--- | :--- | :--- | :--- |
| **en** | Inglês | 82.27 | **Língua Principal** | Manter |
| **de** | Alemão | 1.59 | Possível ruído ou nicho | Revisar / Filtrar |
| **pt** | Português | 1.14 | **Língua Secundária** | Manter |
| **af** | Africâner | 1.08 | Provável erro (falso positivo) | Filtrar / Descartar |
| **tl** | Tagalo | 1.06 | Provável erro (falso positivo) | Filtrar / Descartar |
| **da** | Dinamarquês | 1.04 | Possível ruído | Filtrar / Descartar |
| **es** | Espanhol | 0.98 | Possível ruído | Revisar |
| **nl** | Holandês | 0.87 | Possível ruído | Filtrar / Descartar |
| **fr** | Francês | 0.86 | Possível ruído | Filtrar / Descartar |
| **it** | Italiano | 0.82 | Possível ruído | Filtrar / Descartar |
| **no** | Norueguês | 0.72 | Possível ruído | Filtrar / Descartar |
| **unknown**| Desconhecido | 0.71 | Emojis, links ou números | Descartar |
| **id** | Indonésio | 0.69 | Provável erro (falso positivo) | Filtrar / Descartar |
| **cy** | Galês | 0.67 | Erro clássico de textos curtos | Filtrar / Descartar |
| **so** | Somali | 0.64 | Erro clássico de textos curtos | Filtrar / Descartar |

---

In [10]:
lang_map = df.set_index('id')['lang'].to_dict()

for depth in range(3 + 1):
    dir_ = f"../data/raw/expansao/{depth}/subreddits"
    if not os.path.exists(dir_): continue
    
    for filename in os.listdir(dir_):
        if filename.endswith(".json"):
            path = os.path.join(dir_, filename)
            try:
                with open(path, "r", encoding="utf-8") as f:
                    posts = json.load(f)
                
                for post in posts:
                    post["lang"] = lang_map.get(post["id"], "unknown")
                
                with open(path, "w", encoding="utf-8") as f:
                    json.dump(posts, f, indent=4, ensure_ascii=False)
                    
            except Exception as e:
                print(f"Erro ao salvar {path}: {e}")

In [16]:
df[['id', 'text']].to_csv("../data/processed/dataset_limpo.csv", index=False)